# 04 — Transfer Entropy

Close the classical-information arc: make information flow **directed** — who drives whom — by naming the axes of `02/03`'s conditional mutual information by time, compute it by hand on a system whose true table you can write down, see exactly why the conditioning matters (lagged mutual information reports a backward flow that does not exist; transfer entropy does not), and then earn the honesty rules every estimate from signals needs: the plug-in bias, the surrogate that preserves autocorrelation, the lag, the history length, and the synergy that no pairwise measure can see. This is the measure most used to probe directed coupling in interacting systems — the kind of question hyperscanning asks of two brains — and it is the last classical stone before the quantum lifts.

**Prerequisites:** `02/03_mutual_information` (conditional mutual information and its four-entropy form, the chain rule, the data-processing inequality, the plug-in bias and the shuffled-pairs surrogate), `02/01_shannon_entropy` (conditional entropy, the chain rule), `01/19_quantum_channels` (a stochastic matrix as a noisy step in time — a Markov chain).

**What you'll be able to do**
- Define transfer entropy $\mathrm{TE}_{Y\to X} = I(X_{t+1};\,Y_t\,|\,X_t)$ and rewrite it by hand as $H(X_{t+1}|X_t) - H(X_{t+1}|X_t, Y_t)$, the reduction in the target's next-step uncertainty that the source's past buys *beyond the target's own past*.
- Compute it from the exact table of a lagged-copy system ($1$ bit forward, $0$ backward) and of a noisy copy ($1 - h(\epsilon)$ — `02/03`'s flip channel, now in time), then from simulated data with the course helper, and compare.
- Show on an autocorrelated source why conditioning is the whole point: lagged mutual information finds $0.20$ bits of "backward flow" where there is none; transfer entropy finds $0$.
- Measure the plug-in bias of the estimator, match it to $\frac{|Z|(|X|-1)(|Y|-1)}{2N\ln 2}$, and run the surrogate test correctly — a naive shuffle destroys autocorrelation and flags about half of the independent autocorrelated pairs as coupled when it tests lagged mutual information; circular shifts give close to the nominal $5\%$.
- State the limits honestly: lag and history length are choices, synergy (XOR) is invisible to pairwise transfer entropy, partial information decomposition has no agreed definition, and for Gaussian signals transfer entropy *is* Granger causality.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.infotheory.classical import conditional_mutual_information, mutual_information, shannon_entropy, transfer_entropy

rng = np.random.default_rng(0)   # simulated signals and surrogates throughout
viz.use_course_style()

def h(p):
    return shannon_entropy([p, 1 - p])   # binary entropy of 02/01

def H(P):
    return shannon_entropy(np.asarray(P).ravel())

## 1. Name the axes by time

Take two signals sampled in discrete time, $X_t$ and $Y_t$ (here: binary symbols). Mutual information between them is symmetric and timeless. Schreiber's move (2000) is to ask a *conditional* question with a direction built in:

$$
\mathrm{TE}_{Y\to X} \;=\; I\big(X_{t+1};\,Y_t\,\big|\,X_t\big) ,
$$

how much the source's present $Y_t$ tells you about the target's *next* value $X_{t+1}$, **beyond what the target's own present $X_t$ already tells you**. Using `02/03`'s four-entropy form of conditional mutual information with $(X, Y, Z) = (X_{t+1}, Y_t, X_t)$, and the chain rule $H(AB) = H(B) + H(A|B)$ twice, this is, by hand,

$$
\mathrm{TE}_{Y\to X} = \big[H(X_{t+1}X_t) - H(X_t)\big] - \big[H(X_{t+1}Y_tX_t) - H(Y_tX_t)\big] = H(X_{t+1}|X_t) - H(X_{t+1}|X_t, Y_t) ,
$$

the *reduction in the target's next-step uncertainty* bought by the source's past. It is $\ge 0$ (a conditional mutual information), and it is **not symmetric**: swapping the roles gives $I(Y_{t+1}; X_t\,|\,Y_t)$, a different table. Stationarity is assumed (the joint distribution of $(X_{t+1}, Y_t, X_t)$ does not depend on $t$), and this notebook uses one symbol of history on each side and a lag of one step — the helper `transfer_entropy(source, target)` does exactly that: it builds the $3$-D table of $(X_{t+1}, Y_t, X_t)$ from counts and calls `conditional_mutual_information` on axes $(X, Y, Z) = (\text{target}_{t+1}, \text{source}_t, \text{target}_t)$.

Let's start where the true table is known. **Lagged copy:** $Y_t$ is a fair coin at every step, independent of everything before, and $X_{t+1} = Y_t$. Then $X_t = Y_{t-1}$ is independent of $X_{t+1} = Y_t$, so $H(X_{t+1}|X_t) = 1$; given $Y_t$ the next $X$ is certain, $H(X_{t+1}|X_t, Y_t) = 0$; hence $\mathrm{TE}_{Y\to X} = 1$ bit. Backward: $Y_{t+1}$ is a fresh coin whatever you know, so $\mathrm{TE}_{X\to Y} = 0$. Write the exact $3$-D table $p(x_{t+1}, y_t, x_t)$ — it has four equal entries of $\tfrac14$ where $x_{t+1} = y_t$ and zeros elsewhere — feed it to `02/03`'s four-entropy form, then simulate the system and let the helper count.


In [ ]:
# exact table of the lagged copy: p(x_{t+1}, y_t, x_t) = 1/4 if x_{t+1} == y_t else 0   (axes: X_{t+1}, Y_t, X_t)
P_copy = np.zeros((2, 2, 2))
for y_t in range(2):
    for x_t in range(2):
        P_copy[y_t, y_t, x_t] = 0.25
def te_from_table(P):          # TE = H(X_{t+1} | X_t) - H(X_{t+1} | X_t, Y_t), all from the 3-D table (axes X_{t+1}, Y_t, X_t)
    H_next_and_past = H(P.sum(axis=1))          # (X_{t+1}, X_t)
    H_past = H(P.sum(axis=(0, 1)))              # X_t
    H_all = H(P)
    H_past_and_source = H(P.sum(axis=0))        # (Y_t, X_t)
    return (H_next_and_past - H_past) - (H_all - H_past_and_source)
print(f"lagged copy, exact table:  TE(Y -> X) = {te_from_table(P_copy):.4f} bits   (helper on the same table: {conditional_mutual_information(P_copy):.4f})")
# the backward table p(y_{t+1}, x_t, y_t): y_{t+1} is a fresh coin -> product of (1/2) with the (x_t, y_t) table, which is itself uniform
P_back = np.full((2, 2, 2), 0.125)
print(f"lagged copy, exact table:  TE(X -> Y) = {te_from_table(P_back):.4f} bits")

# now from data: simulate and count
N = 20_000
source = rng.integers(0, 2, size=N)
target = np.empty_like(source); target[0] = 0; target[1:] = source[:-1]     # X_{t+1} = Y_t
counts = np.zeros((2, 2, 2)); np.add.at(counts, (target[1:], source[:-1], target[:-1]), 1.0)   # the 3-D count table by hand: (X_{t+1}, Y_t, X_t)
te_fwd, te_bwd = transfer_entropy(source, target), transfer_entropy(target, source)
print(f"lagged copy, {N} samples:  TE(source -> target) from the hand-built count table = {te_from_table(counts / counts.sum()):.4f};  helper = {te_fwd:.4f};  TE(target -> source) = {te_bwd:.4f}   (counts, lag 1, one symbol of history)")
print(f"for contrast, mutual information at lag 1, both directions: {mutual_information(np.histogram2d(source[:-1], target[1:], bins=2)[0]):.4f} and {mutual_information(np.histogram2d(target[:-1], source[1:], bins=2)[0]):.4f}")

**Read the output.** The exact table gives $1$ bit forward and $0$ backward, and the helper agrees on the same table. Twenty thousand samples reproduce it to four decimals: $\mathrm{TE}_{Y\to X} = 1.000$, $\mathrm{TE}_{X\to Y} = 0.000$ — the direction of the copy, read off the counts. Mutual information at lag one already tells the two directions apart here ($1$ bit source$\to$target, $0$ the other way) because the source has no memory of its own; the next section builds the case where it does not, and that case is the reason transfer entropy conditions on the target's past.


## 2. A noisy copy, and why the conditioning is the whole point

**Noisy copy.** Let $X_{t+1} = Y_t \oplus n_t$, a fair-coin source copied with a flip of probability $\epsilon$ — `02/03`'s binary symmetric channel, now acting across one time step. With $Y_t$ fresh at each step, $H(X_{t+1}|X_t) = 1$ and $H(X_{t+1}|X_t, Y_t) = h(\epsilon)$, so $\mathrm{TE}_{Y\to X} = 1 - h(\epsilon)$: the same U as `02/03` §2, now a transfer. The backward direction stays $0$. We sweep $\epsilon$ and compare counts with the closed form.

**A source with memory.** Now give the source a memory: $Y$ is a sticky coin, $Y_{t+1} = Y_t$ with probability $a = 0.9$ (else it flips) — a two-state Markov chain, the stochastic matrix of `01/19` stepping in time — and the target is a noisy copy, $X_{t+1} = Y_t \oplus n_t$, $\epsilon = 0.1$. Nothing flows from $X$ to $Y$: $Y$'s future depends on $Y$'s present only. But look at the *lagged mutual information* $I(Y_{t+1}; X_t)$: $X_t = Y_{t-1}\oplus n$, and $Y_{t+1}$ agrees with $Y_{t-1}$ after two sticky steps with probability $a^2 + (1-a)^2 = 0.82$, so $Y_{t+1}$ agrees with $X_t$ with probability $p = 0.82\cdot0.9 + 0.18\cdot0.1 = 0.756$ — lagged mutual information reports $1 - h(0.756) = 0.198$ bits of "flow" from $X$ to $Y$ that is entirely $Y$'s own autocorrelation, leaking through the copy. The leak is bounded by the data-processing inequality of `02/03` §3: $X_t$ is a processing of $Y_{t-1}$ (the flip channel), and $Y_{t+1} \leftarrow Y_{t-1} \to X_t$ is a chain, so $I(Y_{t+1}; X_t) \le I(Y_{t+1}; Y_{t-1}) = 1 - h(0.82) = 0.320$ bits — the leak can never exceed what the source remembers of itself two steps back. Transfer entropy conditions on $Y_t$, and given $Y_t$ the chain's next step forgets $Y_{t-1}$ — hence $X_t$ — so $\mathrm{TE}_{X\to Y} = 0$. Forward, by hand: $X_{t+1} = Y_t\oplus n_t$ and $X_t = Y_{t-1}\oplus n_{t-1}$ agree iff the source stayed and the two flips agreed, or the source moved and the flips differed: $q = a\,(\epsilon^2 + (1-\epsilon)^2) + (1-a)\,2\epsilon(1-\epsilon) = 0.9\cdot0.82 + 0.1\cdot0.18 = 0.756$ — the same number as $p$, but only because $a = 1 - \epsilon$ here (the two $0.82$'s are different quantities: $a^2 + (1-a)^2$ and $\epsilon^2 + (1-\epsilon)^2$); exercise 4 separates them. So $H(X_{t+1}|X_t) = h(0.756) = 0.802$, and $H(X_{t+1}|X_t, Y_t) = h(0.1) = 0.469$: $\mathrm{TE}_{Y\to X} = 0.333$ bits — smaller than the memoryless $0.531$, because the target's own past already predicts part of its future. Let's simulate both systems and compare all four numbers with these closed forms.


In [ ]:
def sticky_chain(n, a, rng):
    y = np.empty(n, dtype=int); y[0] = rng.integers(0, 2)
    stay = rng.random(n) < a
    for t in range(1, n):
        y[t] = y[t - 1] if stay[t] else 1 - y[t - 1]
    return y

def noisy_copy(src, eps, rng):
    x = np.empty_like(src); x[0] = 0
    x[1:] = src[:-1] ^ (rng.random(len(src) - 1) < eps).astype(int)
    return x

def lagged_mi(src, tgt):       # I(tgt_{t+1}; src_t)
    counts = np.zeros((2, 2)); np.add.at(counts, (src[:-1], tgt[1:]), 1.0)
    return mutual_information(counts / counts.sum())

# (a) noisy copy of a memoryless source: TE = 1 - h(eps)
eps_grid = np.linspace(0.0, 1.0, 21)
te_eps = []
for e in eps_grid:
    src = rng.integers(0, 2, size=N); tgt = noisy_copy(src, e, rng)
    te_eps.append(transfer_entropy(src, tgt))
te_eps = np.array(te_eps)
closed = np.array([1 - h(e) if 0 < e < 1 else 1.0 for e in eps_grid])
# (b) sticky source, noisy copy: four numbers
a, eps = 0.9, 0.1
Y = sticky_chain(N, a, rng); X = noisy_copy(Y, eps, rng)
p_agree = (a**2 + (1 - a)**2) * (1 - eps) + (1 - a**2 - (1 - a)**2) * eps     # P(Y_{t+1} == X_t): two sticky steps, one flip
q_agree = a * (eps**2 + (1 - eps)**2) + (1 - a) * 2 * eps * (1 - eps)          # P(X_{t+1} == X_t): one sticky step, two flips
rows = [("TE  Y -> X", transfer_entropy(Y, X), h(q_agree) - h(eps)),
        ("TE  X -> Y", transfer_entropy(X, Y), 0.0),
        ("lagged MI  Y -> X", lagged_mi(Y, X), 1 - h(eps)),
        ("lagged MI  X -> Y", lagged_mi(X, Y), 1 - h(p_agree))]

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4), gridspec_kw={"wspace": 0.3})
ax = axes[0]
ax.plot(eps_grid, closed, color=viz.TARGET_COLOR, lw=1.4, ls="--", label="1 - h(eps)")
ax.scatter(eps_grid, te_eps, color=viz.SOURCE_COLOR, s=28, zorder=5, label=f"TE(source -> target), {N} samples")
ax.set_xlabel("flip probability eps of the copy"); ax.set_ylabel("bits")
ax.set_title("Noisy copy of a memoryless source", pad=10); ax.legend(loc="upper center")
ax = axes[1]
names = [r[0] for r in rows]; est = [r[1] for r in rows]; exact = [r[2] for r in rows]
xpos = np.arange(4)
ax.bar(xpos - 0.18, est, width=0.36, color=viz.FLOW_COLOR, label="from counts")
ax.bar(xpos + 0.18, exact, width=0.36, color=[viz.TARGET_COLOR] * 4, alpha=0.8, label="closed form")
ax.set_xticks(xpos); ax.set_xticklabels(names, rotation=15); ax.set_ylabel("bits")
ax.set_ylim(0, 0.68); ax.set_title(f"Sticky source (a = {a}), noisy copy (eps = {eps})", pad=10); ax.legend(loc="upper left")
plt.show()

print(f"noisy copy: max |TE - (1 - h(eps))| over the sweep = {np.max(np.abs(te_eps - closed)):.4f} bits")
print(f"sticky source: agreement probabilities p = P(Y_t+1 == X_t) = {p_agree:.3f},  q = P(X_t+1 == X_t) = {q_agree:.3f}   (equal here because a = 1 - eps)")
print(f"data-processing bound on the backward leak: I(Y_t+1; X_t) = {1 - h(p_agree):.4f} <= I(Y_t+1; Y_t-1) = 1 - h(a^2 + (1-a)^2) = {1 - h(a**2 + (1 - a)**2):.4f}")
for name, e_, c_ in rows:
    print(f"  {name:18s} counts = {e_:.4f}   closed form = {c_:.4f}")

**Read the figure and the output.** Left: the transfer entropy of a noisy copy follows $1 - h(\epsilon)$ across the whole sweep (largest miss $0.012$ bits — the sampling error of $20{,}000$ counts, about a hundredth of a bit here): $1$ bit at a perfect copy, $0$ at $\epsilon = \tfrac12$, and $1$ bit again at $\epsilon = 1$ — a reliably *inverted* copy transfers as much as a faithful one; transfer entropy, like mutual information, does not care about the sign. Right: the four numbers of the sticky system, each within about a hundredth of a bit of its closed form. Transfer entropy says what is true — $0.333$ bits from $Y$ to $X$, **zero** from $X$ to $Y$. Lagged mutual information says $0.531$ and **$0.198$**: it manufactures a backward flow out of the source's own autocorrelation, because $X_t$ remembers $Y_{t-1}$, and $Y_{t-1}$ predicts $Y_{t+1}$ — a leak that the data-processing bound caps at $0.320$ bits. Conditioning on the receiver's own present removes exactly that leak: given $Y_t$, $Y_{t-1}$ has nothing more to say about $Y_{t+1}$ — the Markov property of `02/03` §3, used as a tool. This is the whole reason to condition, and it is the trap any "lagged correlation" analysis of two autocorrelated signals walks into. Note also that $\mathrm{TE}_{Y\to X}$ dropped from $0.531$ to $0.333$ when the source became sticky: transfer entropy credits the source only with what the target's own past could *not* predict.


## 3. From signals: the bias, and the surrogate that respects time

Everything above used $20{,}000$ samples and known tables. From a recording you have fewer samples, no table, and — this is the new difficulty — **autocorrelated** signals. Two things follow.

**Bias.** The estimator is a plug-in conditional mutual information, so `02/03` §4 applies with one more factor: the $\chi^2$ limit now has $|Z|\,(|X|-1)(|Y|-1)$ degrees of freedom (one independent $2\times2$ table per value of the conditioner), so on *independent* signals

$$
\mathbb E\big[\widehat{\mathrm{TE}}\big] \approx \frac{|Z|\,(|X|-1)(|Y|-1)}{2N\ln 2} = \frac{1}{N\ln 2}\ \text{bits}
$$

for binary signals with one symbol of history (leading order, well-populated table) — and it grows fast with history length $k$ ($|Z| = 2^k$) and with the alphabet (binning a continuous signal into $m$ levels: $m^k(m-1)^2$). Always positive on independent data. Let's check it, then face the second problem.

**Surrogates, and autocorrelation.** `02/03` tested an estimate against the *shuffled-pairs* null — random re-pairings of $x$ and $y$, which destroy dependence and keep the marginals. On *signals* that shuffle also destroys each signal's **autocorrelation**, and that changes the null: an autocorrelated signal has fewer *effectively independent* samples than $N$, so its plug-in bias is larger than the shuffle's. The consequence is a calibrated-looking test that is not: two *independent* sticky signals, each with $a = 0.95$, give a lagged mutual information whose true null distribution sits far above the shuffled one. The honest null keeps each signal intact and breaks only their *alignment in time* — **circular shifts** (roll one signal by a random offset) or, across repeated trials, trial shuffles; phase-randomised surrogates (Theiler et al. 1992) also preserve the autocorrelation but impose a *linear-Gaussian* null on top, so a rejection may mean nonlinearity rather than coupling — for transfer entropy prefer the shifts or, as Vicente et al. (2011) do, trial shuffles. And for a *dyad* recorded together, no within-pair surrogate removes the common-input confound (a shared stimulus drives both): that needs pseudo-pairs, signals from two people who never met. Transfer entropy is more robust here — conditioning on the target's own past absorbs most of the target's autocorrelation — but the rule is the same, and the experiment shows both. We run $500$ independent pairs of sticky signals of length $500$, estimate lagged mutual information and transfer entropy, and compare the real null distribution with the shuffle null and the circular-shift null.


In [ ]:
# bias of the TE estimator on independent memoryless bits: |Z|(|X|-1)(|Y|-1) / (2 N ln 2) = 1 / (N ln 2)
print(f"{'N':>6s} {'mean TE_hat (indep.)':>22s} {'1/(N ln2)':>10s} {'95th pct':>9s}")
for n in [100, 400, 1600]:
    ests = np.array([transfer_entropy(rng.integers(0, 2, n), rng.integers(0, 2, n)) for _ in range(1000)])
    print(f"{n:6d} {ests.mean():22.5f} {1 / (n * np.log(2)):10.5f} {np.percentile(ests, 95):9.5f}")

# surrogates on autocorrelated but INDEPENDENT signals
# ten independent batches of 500 pairs: the spread of a false-positive rate across batches is its sampling noise
n, a_null, trials, batches = 500, 0.95, 500, 10
fp_rates = {key: [] for key in ("lagged MI, shuffle", "lagged MI, circular shift", "TE, shuffle", "TE, circular shift")}
for b in range(batches):
    real_mi, shuf_mi, circ_mi = [], [], []
    real_te, shuf_te, circ_te = [], [], []
    for _ in range(trials):
        ys, xs = sticky_chain(n, a_null, rng), sticky_chain(n, a_null, rng)     # independent, both sticky
        shift = rng.integers(20, n - 20)
        real_mi.append(lagged_mi(ys, xs)); shuf_mi.append(lagged_mi(rng.permutation(ys), xs)); circ_mi.append(lagged_mi(np.roll(ys, shift), xs))
        real_te.append(transfer_entropy(ys, xs)); shuf_te.append(transfer_entropy(rng.permutation(ys), xs)); circ_te.append(transfer_entropy(np.roll(ys, shift), xs))
    real_mi, shuf_mi, circ_mi, real_te, shuf_te, circ_te = map(np.array, (real_mi, shuf_mi, circ_mi, real_te, shuf_te, circ_te))
    for key, real, surr in [("lagged MI, shuffle", real_mi, shuf_mi), ("lagged MI, circular shift", real_mi, circ_mi),
                            ("TE, shuffle", real_te, shuf_te), ("TE, circular shift", real_te, circ_te)]:
        fp_rates[key].append(np.mean(real > np.percentile(surr, 95)))   # threshold estimated from the batch's own 500 surrogates
    if b == 0:
        first = (real_mi, shuf_mi, circ_mi, real_te, shuf_te, circ_te)   # the figure shows the first batch
real_mi, shuf_mi, circ_mi, real_te, shuf_te, circ_te = first

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4), gridspec_kw={"wspace": 0.28})
for ax, (real, shuf, circ, title) in zip(axes, [(real_mi, shuf_mi, circ_mi, "lagged mutual information"), (real_te, shuf_te, circ_te, "transfer entropy")]):
    hi = np.percentile(np.concatenate([real, circ, shuf]), 99)       # clip the long right tail for legibility
    bins = np.linspace(0, hi, 40)
    n_real, _, _ = ax.hist(real, bins=bins, color=COLORS["muted"], alpha=0.9, label="true null (independent signals)")
    ax.hist(shuf, bins=bins, histtype="step", lw=2, color=viz.SOURCE_COLOR, label="shuffle surrogate (peak may be off-scale)")
    ax.hist(circ, bins=bins, histtype="step", lw=2, color=viz.TARGET_COLOR, label="circular-shift surrogate")
    ax.set_ylim(0, 1.6 * n_real.max())                                   # keep the true null legible; the shuffle spike is clipped
    ax.axvline(np.percentile(shuf, 95), color=viz.SOURCE_COLOR, ls="--", lw=1)
    ax.axvline(np.percentile(circ, 95), color=viz.TARGET_COLOR, ls="--", lw=1)
    ax.set_xlabel("estimate (bits)   (axis clipped at the 99th percentile)"); ax.set_ylabel("count of pairs"); ax.set_title(title, pad=10)
    ax.legend(loc="upper right")
fig.suptitle(f"Two INDEPENDENT sticky signals (a = {a_null}, N = {n}), first batch of {trials} pairs: the null an estimate must beat, and two surrogates for it", y=1.02)
plt.show()

print(f"first batch of {trials} pairs:")
for name, real, shuf, circ in [("lagged MI", real_mi, shuf_mi, circ_mi), ("TE", real_te, shuf_te, circ_te)]:
    print(f"{name:10s} true null: mean {real.mean():.4f}, 95th pct {np.percentile(real, 95):.4f} | shuffle null 95th {np.percentile(shuf, 95):.4f} -> false positives {np.mean(real > np.percentile(shuf, 95)):.3f}"
          f" | circular-shift null 95th {np.percentile(circ, 95):.4f} -> false positives {np.mean(real > np.percentile(circ, 95)):.3f}")
print(f"\nfalse-positive rate at the 5% level over {batches} batches of {trials} pairs (nominal 0.050):")
for key, rates in fp_rates.items():
    rates = np.array(rates)
    print(f"  {key:26s} mean {rates.mean():.3f}   standard deviation {rates.std(ddof=1):.3f}   range {rates.min():.3f} to {rates.max():.3f}")

**Read the figure and the output.** The bias on independent bits tracks $\frac{1}{N\ln 2}$ — $0.0149$ at $N = 100$, $0.0037$ at $400$, $0.0009$ at $1600$ — always positive, as it must be. Then the trap, on the left: for two *independent* sticky signals the true null distribution of lagged mutual information (grey) is wide, with a $95$th percentile near $0.06$ bits, while the shuffle surrogate (periwinkle outline) collapses to a sliver near $0.006$: testing against the shuffle calls **about half** of the independent pairs "significant" at the $5\%$ level: $53\%$ on average over ten batches of $500$ pairs, from $48\%$ to $58\%$. The circular-shift surrogate (amber outline) reproduces the true null almost exactly and gives the false-positive rate it promises, $4.9\%$ on average, between $3.4\%$ and $6.4\%$. The mechanism is effective sample size: a signal that stays put for twenty steps at a time carries far fewer independent samples than $N = 500$, so its plug-in bias is far larger than that of a shuffled copy of itself. On the right, transfer entropy is much better behaved — its true null, shuffle null and circular-shift null nearly coincide, because conditioning on $X_t$ absorbs the *target's* autocorrelation — and both surrogates give the nominal rate on average, $4.9\%$ for the shuffle and $5.0\%$ for the circular shift. One batch can mislead (the first, in the figure, gives $9\%$ and $8\%$), so with $500$ pairs this experiment cannot tell the two surrogates apart for transfer entropy. (One remark on procedure: this experiment calibrates at the population level — one surrogate per pair, pooled — which has the same expectation as, but is not the same procedure as, the per-recording test a practitioner runs with hundreds of surrogates of *one* pair.) The rule from here to the end of the course: **an estimated coupling is a number relative to a surrogate distribution that preserves each signal's own structure and destroys only their alignment.** `02/03`'s shuffle was correct for independent *samples*; signals are not samples.


## 4. What transfer entropy cannot tell you

Three limits, each shown on a system you can write down, so you know them before you meet them in data.

**Lag and history are choices.** The helper looks one step back with one symbol of history. If the true influence arrives two steps later ($X_{t+1} = Y_{t-1}$, copied here with a flip of probability $0.1$), a lag-$1$ estimate is neither the full value nor $0$ — it sees part of the copy through whatever autocorrelation the source has ($Y_t$ predicts $Y_{t-1}$ when the source is sticky; with a memoryless source it would see nothing at all) — and the right move is to *scan* the lag (and the history length $k$, at the cost of $2^k$ conditioning cells and the bias of section 3). In practice: a scan over lags with the maximum reported as the estimated delay (Wibral et al. 2013), and a history length and delay chosen by a stated criterion or a scan (Vicente et al. 2011) — both reported as choices.

**Two-way coupling.** Real systems rarely drive in one direction only. With $X$ copying $Y$ $80\%$ of the time and $Y$ copying $X$ $30\%$ of the time, both transfer entropies are positive; the *difference* (net flow) names the dominant direction, and the smaller one is real, not noise — test each against its own surrogate null.

**Synergy, and the decomposition that has no agreed definition.** Let $X_{t+1} = A_t \oplus B_t$ with $A, B$ independent fair coins. Neither source alone carries anything about $X_{t+1}$ — both pairwise transfer entropies are $0$ — yet the two together determine it exactly, $\mathrm{TE}_{(A,B)\to X} = 1$ bit. Pairwise measures are blind to **synergy**. The natural wish is to split a joint information into *redundant*, *unique* and *synergistic* parts — partial information decomposition (Williams & Beer 2010) — but the redundancy term admits several inequivalent definitions (Williams–Beer $I_{\min}$, Bertschinger et al. 2014, Ince 2017, …) with no consensus; treat any reported synergy/redundancy as definition-dependent. Let's see the three limits in numbers.


In [ ]:
# (i) the wrong lag: a noisy lag-2 copy of a sticky source, read at lag 1; then a scan over lags
src = sticky_chain(N, 0.9, rng)
tgt2 = np.zeros_like(src); tgt2[2:] = src[:-2] ^ (rng.random(N - 2) < 0.1).astype(int)   # X_{t+1} = Y_{t-1} xor flip, eps = 0.1
print(f"noisy lag-2 copy of a sticky source, read at lag 1:  TE = {transfer_entropy(src, tgt2):.4f} bits   (neither 0 nor the value at the true lag)")
for k in [1, 2, 3, 4]:
    te_k = transfer_entropy(src[:len(src) - (k - 1)], tgt2[k - 1:])   # shift the source back by k-1 extra steps
    print(f"  scan: lag {k}  ->  TE = {te_k:.4f}")
# (ii) two-way coupling
Xb = np.zeros(N, dtype=int); Yb = np.zeros(N, dtype=int)
u = rng.random((N, 2)); fresh = rng.integers(0, 2, size=(N, 2))
for t in range(1, N):
    Xb[t] = Yb[t - 1] if u[t, 0] < 0.8 else fresh[t, 0]              # X copies Y 80% of the time
    Yb[t] = Xb[t - 1] if u[t, 1] < 0.3 else fresh[t, 1]              # Y copies X 30% of the time
te_yx, te_xy = transfer_entropy(Yb, Xb), transfer_entropy(Xb, Yb)
null_xy = np.array([transfer_entropy(np.roll(Xb, rng.integers(20, N - 20)), Yb) for _ in range(200)])
print(f"two-way coupling:  TE(Y -> X) = {te_yx:.4f}   TE(X -> Y) = {te_xy:.4f}   net Y -> X = {te_yx - te_xy:+.4f};  the smaller one vs its circular-shift null 95th pct {np.percentile(null_xy, 95):.4f}: real")
# (iii) synergy: XOR
A_ = rng.integers(0, 2, size=N); B_ = rng.integers(0, 2, size=N)
Xs = np.zeros(N, dtype=int); Xs[1:] = A_[:-1] ^ B_[:-1]
print(f"XOR target:  TE(A -> X) = {transfer_entropy(A_, Xs):.4f}   TE(B -> X) = {transfer_entropy(B_, Xs):.4f}   TE((A,B) -> X) = {transfer_entropy(2 * A_ + B_, Xs):.4f}   (joint source as a 4-symbol signal)")

**Read the output.** A noisy lag-$2$ copy of a sticky source read at lag $1$ gives $0.17$ bits: the coupling is not at lag $1$, and yet the value is far from $0$, because $Y_t$ predicts $Y_{t-1}$. The scan finds the maximum at the true lag $2$, $0.33$ bits — section 2's $0.333$, since read at its true lag this is section 2's system — and smaller values on both sides, $0.12$ at lag $3$ and $0.05$ at lag $4$: the sticky source makes neighbouring values predictive, and the target's noisy past does not absorb them. The lag is a parameter you scan and report, not a default you trust. In the two-way system both transfer entropies are positive, $0.55$ and $0.07$; the net flow names $Y\to X$ as dominant, and the smaller one clears its own circular-shift null by a wide margin — a weak coupling is still a coupling. And the XOR target: $0$ from $A$, $0$ from $B$, $1$ bit from the pair — synergy that no pairwise analysis will ever see, and the reason decomposition is tempting; keep in mind that the split it offers is not unique.


## 5. Closing the classical arc

Four notebooks, one idea each, each built from the one before: surprise and entropy (`02/01`), the cost of a wrong model (`02/02`), the bits two variables share (`02/03`), and today the bits one signal's past adds to another's future beyond its own. Two remarks place it in the rest of the course.

- **Granger causality is the Gaussian case.** For jointly Gaussian (linear) signals, Granger's statistic is the log-ratio of residual variances of the restricted and the full autoregressive fits, $F = \ln(\sigma^2_{\text{restricted}}/\sigma^2_{\text{full}})$, and transfer entropy in nats is *exactly* $F/2$ (in bits, $\tfrac12\log_2$ of the ratio) — Barnett, Barrett & Seth 2009. Transfer entropy is the model-free generalisation; Granger is what you get when the tables are Gaussians.
- **Direction does not lift.** The quantum coupling measures of module 04 — quantum mutual information (`02/10`), the Bures distance (`02/12`), the quantum transport cost on an embedded pair — are **symmetric**. When a question is about direction, it stays a classical question asked of the raw signals with today's tool and today's surrogates. That is why the dictionary row below has no quantum partner, and says so.


## Your turn

1. **Noisy copy, your surrogate.** Build a noisy copy with $\epsilon = 0.3$ and $N = 2000$, estimate $\mathrm{TE}_{Y\to X}$, and test it against $200$ circular-shift surrogates; then reduce $N$ to $100$ and repeat: at what $N$ does the estimate stop clearing the null? Compare with $1 - h(0.3)$ and with the bias formula of section 3.
2. **Scan the lag.** Build a lag-$3$ noisy copy ($\epsilon = 0.2$) of a sticky source ($a = 0.8$) and compute $\mathrm{TE}$ for lags $1$ to $5$ by shifting the source as in section 4; plot the scan. Why is the lag-$1$ and lag-$2$ value not zero here, unlike the memoryless case?
3. **Net flow.** In the two-way system of section 4, vary the two copy probabilities on a grid $(0.1, \dots, 0.9)^2$ and draw $\mathrm{TE}_{Y\to X} - \mathrm{TE}_{X\to Y}$ as a heatmap (palette from `qot_course.colors`). Where is the net flow zero although both couplings are strong?
4. **Harder: history, bias, and the sticky closed forms.** (a) Derive by hand the four closed forms of section 2 for general $a$ and $\epsilon$ — with the *two* agreement probabilities kept apart, $p = (a^2 + (1-a)^2)(1-\epsilon) + \big(1 - a^2 - (1-a)^2\big)\epsilon$ for the backward lagged mutual information and $q = a(\epsilon^2 + (1-\epsilon)^2) + (1-a)\,2\epsilon(1-\epsilon)$ for $\mathrm{TE}_{Y\to X} = h(q) - h(\epsilon)$ — and check them against counts at $a = 0.7$, $\epsilon = 0.25$, where $p \ne q$. (b) Extend the estimator to $k = 2$ symbols of target history (encode $(X_t, X_{t-1})$ as a $4$-symbol signal and call `conditional_mutual_information` on the $(X_{t+1}, Y_t, \text{history})$ table): on independent bits, confirm the bias becomes $\approx \frac{4}{2N\ln 2}$; on the sticky system, does $\mathrm{TE}_{Y\to X}$ change, and should it? (c) For two independent Gaussian AR(1) signals — $x_t = \phi\,x_{t-1} + \eta_t$ with $\eta_t \sim \mathcal N(0, 1)$ and $\phi = 0.9$, built with a loop like `sticky_chain` — binarised at their median (`x > np.median(x)`), repeat the shuffle-vs-circular-shift experiment of section 3 and report both false-positive rates.


## What you built

- You defined transfer entropy as `02/03`'s conditional mutual information with the axes named by time, rewrote it by hand as $H(X_{t+1}|X_t) - H(X_{t+1}|X_t, Y_t)$, and computed it from an exact table ($1$ bit forward, $0$ back) and from counts.
- You found $1 - h(\epsilon)$ for a noisy copy and, on a sticky source, the four closed forms that show the point of conditioning: lagged mutual information invents $0.198$ bits of backward flow from the source's autocorrelation; transfer entropy finds $0$.
- You measured the estimator's bias ($\frac{1}{N\ln 2}$ for binary, one-step history) and ran the surrogate test the right way: a naive shuffle flags about half of the independent autocorrelated pairs as coupled when it tests lagged mutual information; circular shifts give close to the nominal $5\%$.
- You saw the three limits on systems you can write down — the lag is a choice, two-way coupling needs both directions and their nulls, XOR synergy is invisible pairwise and the decomposition that would expose it is not unique — and placed Granger causality as the Gaussian case.
- You closed the classical-information arc and added a row whose quantum column honestly says: direction does not lift.

That completes the classical information toolbox — entropy, divergence, shared structure, directed flow — each number earned on a table you could write by hand, and each estimate read against the null it deserves.


**New in the dictionary.** The complete table lives in [the course dictionary](../dictionary.ipynb) (`notebooks/dictionary.ipynb`).

| Classical | Quantum |
| --- | --- |
| transfer entropy $\mathrm{TE}_{Y\to X} = I(X_{t+1};\,Y_t\,|\,X_t)$ | no quantum lift in this course: the quantum coupling measures of module 04 are symmetric |

## What's next

So far every quantity compared masses *bin by bin*, ignoring what the bins are. The geometry arc asks a different question: what does the *space* of all distributions look like, and what does it mean to move through it? `02/05_the_probability_simplex` draws that space — the simplex named in `02/01` — and the notebooks after it put two different geometries on it.


## References

- T. Schreiber, *Measuring information transfer*, Phys. Rev. Lett. **85**, 461–464 (2000) — the definition, as a conditional mutual information / conditional relative entropy. doi:10.1103/PhysRevLett.85.461
- T. M. Cover & J. A. Thomas, *Elements of Information Theory*, 2nd ed., Wiley (2006) — §2.5 (conditional mutual information, chain rule), §2.8 (data processing), §4.1–4.2 (entropy rate, Markov chains). doi:10.1002/047174882X
- J. Theiler, S. Eubank, A. Longtin, B. Galdrikian & J. D. Farmer, *Testing for nonlinearity in time series: the method of surrogate data*, Physica D **58**, 77–94 (1992) — surrogates that preserve the autocorrelation. doi:10.1016/0167-2789(92)90102-S
- L. Barnett, A. B. Barrett & A. K. Seth, *Granger causality and transfer entropy are equivalent for Gaussian variables*, Phys. Rev. Lett. **103**, 238701 (2009). doi:10.1103/PhysRevLett.103.238701
- R. Vicente, M. Wibral, M. Lindner & G. Pipa, *Transfer entropy — a model-free measure of effective connectivity for the neurosciences*, J. Comput. Neurosci. **30**, 45–67 (2011) — embedding choices, trial-shuffling surrogates, and the practice on neural signals. doi:10.1007/s10827-010-0262-3
- P. L. Williams & R. D. Beer, *Nonnegative decomposition of multivariate information*, arXiv:1004.2515 (2010); N. Bertschinger, J. Rauh, E. Olbrich, J. Jost & N. Ay, *Quantifying unique information*, Entropy **16**, 2161–2183 (2014) — partial information decomposition and the non-uniqueness of redundancy. doi:10.3390/e16042161
- M. Wibral, N. Pampu, V. Priesemann, F. Siebenhühner, H. Seiwert, M. Lindner, J. T. Lizier & R. Vicente, *Measuring information-transfer delays*, PLoS ONE **8**, e55809 (2013) — the lag scan recovers the true delay. doi:10.1371/journal.pone.0055809
- R. A. A. Ince, *Measuring multivariate redundant information with pointwise common change in surprisal*, Entropy **19**, 318 (2017) — one of the competing redundancy definitions. doi:10.3390/e19070318
- L. Paninski, *Estimation of entropy and mutual information*, Neural Computation **15**, 1191–1253 (2003) — plug-in bias. doi:10.1162/089976603321780272

**Previous:** `notebooks/02_InformationAndGeometry/03_mutual_information.ipynb` · **Next:** `notebooks/02_InformationAndGeometry/05_the_probability_simplex.ipynb`
